<table style="width: 100%; border-collapse: collapse; border: none; background: #fffbeb; border-left: 6px solid #f59e0b; border-radius: 8px; padding: 20px; box-shadow: 0 2px 4px rgba(0,0,0,0.05);">
  <tr style="border: none;">
    <td style="vertical-align: middle; border: none; padding: 15px 20px;">
      <h1 style="margin: 0; color: #78350f; font-size: 2em; font-family: system-ui, -apple-system, sans-serif; font-weight: 800; letter-spacing: -0.02em;">
        💡 Taller: La Panadería, el Grupo de Vecinos y el Libro de la Cooperativa — Flujos, Grafos y Sellos 🥖🧸
      </h1>
      <p style="margin: 6px 0 0 0; color: #b45309; font-size: 1.15em; font-weight: 600; font-family: system-ui, -apple-system, sans-serif;">
        Especialización en Ciencia de Datos | Big Data
      </p>
      <p style="margin: 4px 0 0 0; color: #92400e; font-size: 0.95em; font-family: system-ui, -apple-system, sans-serif;">
        Universidad Santo Tomás — Seccional Tunja
      </p>
    </td>
    <td style="text-align: right; vertical-align: middle; border: none; padding: 15px 20px; width: 30%;">
      <span style="background: #f59e0b; color: #ffffff; padding: 6px 14px; border-radius: 20px; font-size: 0.85em; font-weight: 700; display: inline-block; margin-bottom: 8px;">
        💡 Taller Práctico Evaluativo · Módulo 05
      </span><br>
      <span style="color: #78350f; font-size: 0.85em;">Docente: Santiago A. Zúñiga M.</span><br>
      <a href="mailto:gestorvirtualcienciadatos@ustatunja.edu.co" style="color: #b45309; font-size: 0.8em; text-decoration: none; font-weight: 500;">gestorvirtualcienciadatos@ustatunja.edu.co</a>
    </td>
  </tr>
</table>

<div align="center" style="margin-top: 15px; margin-bottom: 15px;">
  <a href="https://colab.research.google.com/github/sazuniga06/Data-Science-Programming---USTA-Tunja-Repository/blob/main/Big%20Data/homeworks/Para%20Dummies/05_Streaming_Grafos_Tendencias_Hands_On_Dummies.ipynb" target="_parent">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" style="vertical-align: middle;"/>
  </a>
</div>

---
## 🎈 ¿Qué vamos a hacer en este taller?

Tres historias pequeñas de un pueblo de Boyacá (con datos **inventados**) para practicar lo del Módulo 05:

1. 🥖 **Los pedidos de la panadería:** los pedidos llegan por mensaje **tarde y desordenados**. Contarás cuántos se hicieron en cada tanda de 10 minutos **según la hora del sello** y descubrirás **cuánto hay que esperar** para no perder ninguno.
2. 📱 **El grupo de vecinos:** ¿quién es el vecino más importante? No es el que más conocidos tiene, sino aquel a quien **le piden consejo los que a su vez son importantes** (PageRank por rondas).
3. 📒 **El libro de la cooperativa lechera:** cada hoja lleva un **sello** que depende de la hoja anterior. Verás cómo se **delata** una hoja cambiada y por qué esconder el cambio obliga a rehacer todo lo que sigue.
4. ✍️ **Contar con tus palabras** lo que aprendiste.

> 🧸 **Cómo funciona:** cada reto trae un cuadro de código con huecos `...` y pistas 💡. Reemplaza los `...` por lo que corresponda. Debajo hay una **autoverificación** que te orienta con cariño. No traen soluciones: ¡inténtalo tú!

### 📋 Instrucciones

- Completa solo los huecos `...`; no cambies las celdas de preparación ni de autoverificación.
- Al final usa *Restart & Run All*: todo debe correr sin errores y sin `...` pendientes.
- Todo funciona sin internet (datos inventados y sin archivos externos). No se usa *Spark*. Repasa si quieres: [El buzón del pueblo y el chisme que se riega](../../05%20-%20Avances%20y%20Tendencias%20en%20Big%20Data/Para%20Dummies/00_Stream_y_Grafos_Dummies.ipynb) y [la báscula, los tres colegios y el cuaderno con sellos](../../05%20-%20Avances%20y%20Tendencias%20en%20Big%20Data/Para%20Dummies/01_Tendencias_IA_Blockchain_Dummies.ipynb).

In [ ]:
# ============================================================
# Preparación (no modificar): imports y datos inventados
# ============================================================
import copy
import hashlib
from collections import Counter

import networkx as nx

# Reto 1: 12 pedidos de la panadería «El Trigal». Cada uno: (minuto del SELLO, minuto de LLEGADA), en minutos desde las 6:00
pedidos = [(2, 3), (7, 8), (9, 21), (11, 12), (14, 15), (18, 22), (23, 24), (25, 27), (29, 52), (31, 32), (36, 37), (38, 43)]

# Reto 2: a quién le pide consejo cada vecino del grupo de WhatsApp
personas = ["Marina", "Don Pablo", "Lucía", "Tomás", "Sofía", "Hugo"]
pide_consejo_a = {"Tomás": ["Marina", "Lucía"], "Sofía": ["Marina", "Lucía"], "Hugo": ["Marina", "Lucía"],
                  "Lucía": ["Marina"], "Marina": ["Don Pablo"], "Don Pablo": ["Marina"]}

# Reto 3: entregas de leche de la cooperativa, una por hoja del libro
textos_libro = ["Lunes: 120 litros de Finca La Esperanza", "Martes: 95 litros de Finca El Roble",
                "Miércoles: 140 litros de Finca La Esperanza", "Jueves: 80 litros de Finca El Mirador",
                "Viernes: 110 litros de Finca El Roble"]


def sello(texto, sello_anterior):
    """Huella de 8 caracteres de la hoja: depende de su texto Y del sello de la hoja anterior."""
    return hashlib.sha256((sello_anterior + texto).encode()).hexdigest()[:8]


def huella(texto):
    return hashlib.sha256(texto.encode()).hexdigest()[:8]


print("Pedidos:", len(pedidos), "| Vecinos:", len(personas), "| Hojas del libro:", len(textos_libro))

---
## Reto 1: Los pedidos de la panadería y cuánto esperar 🥖

Los pedidos de «El Trigal» llegan por mensaje. Cada uno trae el **minuto del sello** (cuándo se hizo) y anotamos el **minuto de llegada**. Quieres contar los pedidos **por tandas de 10 minutos**: la tanda 0 es del minuto 0 al 9, la tanda 1 del 10 al 19, y así. Una tanda se **cierra** a los `(tanda + 1) * 10` minutos **más la espera** que decidas; lo que llegue después **se pierde**.

**Pasos**
1. `por_sello`: cuántos pedidos hay por tanda **según el sello** (`minuto // 10`). `por_llegada`: lo mismo pero **según la llegada**.
2. Completa `contar_con_espera(espera)`: un pedido cuenta si su `llegada` es menor o igual que el cierre de **su** tanda (la de su sello).
3. `conteos`: cuántos pedidos se cuentan con cada espera de `candidatos = [0, 5, 10, 15, 20, 25]`.
4. `espera_min`: la **menor** espera con la que se cuentan **los 12 pedidos**.

**Puntaje sugerido:** 30 puntos.

In [ ]:
# Casillas que debes llenar (empiezan vacías; no las borres)
por_sello = por_llegada = None
conteos = espera_min = None

# 💡 1) La tanda de un minuto m es m // 10. Contamos cuántos pedidos caen en cada tanda
por_sello = Counter(sello_ // 10 for sello_, llegada in pedidos)
por_llegada = Counter(... // 10 for sello_, llegada in pedidos)          # ¿qué minuto usas aquí: el del sello o el de la llegada?


# 💡 2) La tanda t se cierra en (t + 1) * 10 + espera. El pedido cuenta si llegó a tiempo
def contar_con_espera(espera):
    n = 0
    for sello_, llegada in pedidos:
        cierre = (sello_ // 10 + 1) * 10 + ...
        if llegada <= ...:
            n += 1
    return n


# 💡 3) Probar varias esperas
candidatos = [0, 5, 10, 15, 20, 25]
conteos = [contar_con_espera(e) for e in candidatos]

# 💡 4) La menor espera con la que se cuentan los 12 pedidos
espera_min = next(e for e in candidatos if contar_con_espera(e) == ...)

In [ ]:
# Autoverificación del Reto 1 (no modificar)
assert por_sello is not None and dict(por_sello) == {0: 3, 1: 3, 2: 3, 3: 3}, "Pista 🧸: por_sello cuenta las tandas con el minuto del SELLO (el primer número de cada pedido) // 10. Debe haber 3 pedidos en cada una de las 4 tandas."
assert por_llegada is not None and dict(por_llegada) == {0: 2, 1: 2, 2: 4, 3: 2, 4: 1, 5: 1}, "Pista 🧸: por_llegada usa el minuto de LLEGADA (el segundo número de cada pedido) // 10."
assert dict(por_llegada) != dict(por_sello), "Contar por la llegada da OTRA historia que contar por el sello: revisa que usaste minutos distintos en cada línea."
assert conteos is not None and conteos == [8, 10, 10, 11, 11, 12], f"Pista 🧸: con espera 0 se cuentan 8 pedidos y con espera 25 los 12; la tanda t se cierra en (t + 1) * 10 + espera y un pedido cuenta si llegada <= cierre. Obtuviste {conteos}."
assert espera_min == 25, "Pista 🧸: espera_min es la primera espera de `candidatos` con la que contar_con_espera(e) == 12."
print(f"✅ ¡Reto 1 superado! Sin esperar solo cuentas {conteos[0]} de 12 pedidos; para no perder ninguno hay que esperar {espera_min} minutos (el pedido más demorado tardó 23).")

---
## Reto 2: ¿Quién es el vecino más importante? 📱

En el grupo de WhatsApp, `pide_consejo_a` dice a quién le pide consejo cada vecino. La **importancia** funciona por rondas: todos empiezan con **100 puntos**. En **cada ronda** cada vecino reparte el **85 %** de sus puntos **por partes iguales** entre las personas a quienes pide consejo, y el **15 %** restante se **reparte entre todo el grupo** (porque a veces uno le pregunta a cualquiera). Tras muchas rondas, los puntos dejan de cambiar: esa es la importancia.

**Pasos**
1. Completa `una_ronda(puntos)`: el 15 % ya está hecho; falta repartir el 85 % entre los consejeros de cada vecino.
2. Repite **100 rondas** a partir de 100 puntos por vecino.
3. `mas_importante`: el vecino con más puntos. `directos`: cuántos vecinos le piden consejo directamente a cada uno.
4. Compara a **Don Pablo** con **Lucía**: ¿quién tiene más admiradores directos y quién más importancia?

**Puntaje sugerido:** 30 puntos.

In [ ]:
# Casillas que debes llenar
puntos = mas_importante = directos = None


# 💡 1) El 15 % se reparte entre todos (ya está); el 85 % de cada vecino se reparte entre sus consejeros
def una_ronda(puntos):
    nuevos = {p: 0.15 * sum(puntos.values()) / len(personas) for p in personas}
    for p, consejeros in pide_consejo_a.items():
        for c in consejeros:
            nuevos[c] += ... * puntos[p] / len(...)       # el 85 % de p, dividido entre cuántos consejeros tiene p
    return nuevos


# 💡 2) Todos empiezan con 100 puntos y se repiten 100 rondas
puntos = {p: 100.0 for p in personas}
for ronda in range(...):
    puntos = una_ronda(...)

# 💡 3) El de más puntos, y cuántos le piden consejo directamente (p in c pregunta: ¿p está en la lista de consejeros c?)
mas_importante = max(puntos, key=...)
directos = {p: sum(p in c for c in pide_consejo_a.values()) for p in personas}

In [ ]:
# Autoverificación del Reto 2 (no modificar)
assert mas_importante is not None and directos is not None, "Pista 🧸: completa los huecos y ejecuta la celda de arriba completa (de arriba abajo)."
assert abs(sum(puntos.values()) - 600) < 1e-9, "Pista 🧸: los puntos se reparten, no se crean ni se pierden: siempre suman 600. El 85 % de p se reparte entre sus consejeros con 0.85 * puntos[p] / len(consejeros)."
_t = una_ronda({p: 100.0 for p in personas})
assert abs(_t["Don Pablo"] - (0.15 * 100 + 0.85 * 100)) < 1e-9 and abs(_t["Tomás"] - 15.0) < 1e-9, "Pista 🧸: tras UNA ronda, Don Pablo recibe 15 + 85 (Marina le da todo) = 100 y Tomás, que nadie aconseja, solo recibe sus 15."
_pr = nx.pagerank(nx.DiGraph([(p, c) for p, cs in pide_consejo_a.items() for c in cs]), alpha=0.85)
assert all(abs(puntos[p] - 600 * _pr[p]) < 0.05 for p in personas), "Pista 🧸: repite 100 rondas encadenadas: puntos = una_ronda(puntos). Debe coincidir con PageRank de networkx (por 600)."
assert mas_importante == "Marina", "Pista 🧸: mas_importante = max(puntos, key=puntos.get)."
assert directos == {"Marina": 5, "Don Pablo": 1, "Lucía": 3, "Tomás": 0, "Sofía": 0, "Hugo": 0}, "Pista 🧸: directos cuenta, para cada persona p, en cuántas listas de consejeros aparece."
assert directos["Don Pablo"] < directos["Lucía"] and puntos["Don Pablo"] > puntos["Lucía"], "Don Pablo tiene MENOS admiradores directos que Lucía pero más importancia: su único admirador es Marina."
print(f"✅ ¡Reto 2 superado! Marina manda ({puntos['Marina']:.0f} puntos). Don Pablo tiene {directos['Don Pablo']} admirador y {puntos['Don Pablo']:.0f} puntos; Lucía tiene {directos['Lucía']} y solo {puntos['Lucía']:.0f}.")

---
## Reto 3: El libro de la cooperativa y sus sellos 📒

Cada hoja del libro trae el texto de una entrega de leche y un **sello** calculado con `sello(texto, sello_anterior)`: depende del texto **y** del sello de la hoja anterior (la primera usa `"00000000"`). Si alguien cambia una hoja vieja, su sello deja de cuadrar y también los de **todas las siguientes**.

**Pasos**
1. Completa `armar_libro(textos)` y `primera_hoja_rota(libro)` (devuelve el número de la primera hoja cuyo sello no cuadra, o `None`). Guarda `libro = armar_libro(textos_libro)` y `rota_original`.
2. **Ataque A:** copia el libro y cambia el **texto** de la hoja 2 **sin tocar el sello**. Guarda `libro_a` y `rota_a`.
3. **Ataque B:** parte de `libro_a`, y además **recalcula el sello** de la hoja 2. Guarda `libro_b` y `rota_b`. ¿Dónde se rompe ahora?
4. **Resumen de resúmenes:** `raiz_de_cuatro` combina las huellas de 4 textos en una sola (las de la izquierda y la derecha se juntan por parejas). Guarda `raiz` (de las primeras 4 hojas) y `raiz_cambiada` (cambiando el texto de la hoja 3).

**Puntaje sugerido:** 25 puntos.

In [ ]:
# Casillas que debes llenar
libro = libro_a = libro_b = None
rota_original = rota_a = rota_b = None
raiz = raiz_cambiada = None


# 💡 1) Cada hoja se sella con su texto y el sello anterior
def armar_libro(textos):
    libro, anterior = [], "00000000"
    for texto in textos:
        anterior = sello(..., ...)
        libro.append({"texto": texto, "sello": anterior})
    return libro


def primera_hoja_rota(libro):
    anterior = "00000000"
    for i, hoja in enumerate(libro):
        if hoja["sello"] != sello(hoja["texto"], ...):      # ¿cuál sello debería usar como «anterior»?
            return i
        anterior = hoja["sello"]
    return None


libro = armar_libro(textos_libro)
rota_original = primera_hoja_rota(libro)

# 💡 2) Ataque A: cambiar el texto de la hoja 2 sin tocar su sello
libro_a = copy.deepcopy(libro)
libro_a[2]["texto"] = "Miércoles: 40 litros de Finca La Esperanza"
rota_a = primera_hoja_rota(...)

# 💡 3) Ataque B: además, recalcular el sello de la hoja 2 (con el sello de la hoja 1 como «anterior»)
libro_b = copy.deepcopy(libro_a)
libro_b[2]["sello"] = sello(libro_b[2]["texto"], libro_b[...]["sello"])
rota_b = primera_hoja_rota(libro_b)


# 💡 4) Resumen de resúmenes: huella(huella(a + b) + huella(c + d))
def raiz_de_cuatro(cuatro_textos):
    a, b, c, d = [huella(t) for t in cuatro_textos]
    izq = huella(a + b)
    der = huella(... + ...)                                 # la pareja de la derecha
    return huella(izq + der)


raiz = raiz_de_cuatro(textos_libro[:4])
cambiado = list(textos_libro[:4])
cambiado[2] = "Miércoles: 14 litros de Finca La Esperanza"
raiz_cambiada = raiz_de_cuatro(...)

In [ ]:
# Autoverificación del Reto 3 (no modificar)
assert libro is not None and len(libro) == 5, "Pista 🧸: libro = armar_libro(textos_libro) debe tener 5 hojas. En armar_libro, el sello se calcula con sello(texto, anterior)."
_ant, _ref = "00000000", []
for _t in textos_libro:
    _ant = hashlib.sha256((_ant + _t).encode()).hexdigest()[:8]
    _ref.append(_ant)
assert [h["sello"] for h in libro] == _ref and [h["texto"] for h in libro] == textos_libro, "Pista 🧸: sello(texto, anterior): el primero usa «00000000» y cada uno de los siguientes, el sello de la hoja ANTERIOR."
assert rota_original is None, "Pista 🧸: un libro sin tocar no tiene hojas rotas: primera_hoja_rota debe devolver None."
assert libro_a is not None and libro_a[2]["sello"] == libro[2]["sello"] and libro_a[2]["texto"] != libro[2]["texto"], "Pista 🧸: en el ataque A solo cambia el texto de la hoja 2; el sello queda igual."
assert rota_a == 2, "Pista 🧸: al cambiar el texto de la hoja 2, su sello ya no cuadra: primera_hoja_rota(libro_a) debe dar 2."
assert libro_b is not None and libro_b[2]["sello"] == sello(libro_b[2]["texto"], libro[1]["sello"]), "Pista 🧸: el tramposo recalcula el sello de la hoja 2 con el sello de la hoja 1 como «anterior»."
assert rota_b == 3, "Pista 🧸: si recalcula el sello de la hoja 2, la hoja 3 (que tiene guardado el sello viejo) deja de cuadrar: rota_b debe ser 3."
_h = lambda s: hashlib.sha256(s.encode()).hexdigest()[:8]
_a, _b, _c, _d = [_h(t) for t in textos_libro[:4]]
assert raiz == _h(_h(_a + _b) + _h(_c + _d)), "Pista 🧸: raiz = huella(huella(a + b) + huella(c + d)), donde a, b, c, d son las huellas de los 4 textos."
assert raiz_cambiada is not None and raiz_cambiada != raiz, "Pista 🧸: cambiar un solo texto cambia la raíz: raiz_cambiada = raiz_de_cuatro(cambiado)."
print(f"✅ ¡Reto 3 superado! Ataque A: se rompe la hoja {rota_a}. Ataque B (recalculando su sello): ahora se rompe la hoja {rota_b}. Y un solo cambio cambió la raíz: {raiz} → {raiz_cambiada}.")

---
## Para cerrar: tu reflexión ✍️

**Puntaje sugerido:** 15 puntos (sin código). Escribe **3 a 5 líneas** en la celda de abajo respondiendo:

- En la panadería, ¿qué se **gana** y qué se **pierde** al esperar más? Si el reporte fuera para **cobrar** los pedidos, ¿esperarías más o menos que si fuera para un aviso en la pantalla?
- ¿Por qué Don Pablo, con un solo admirador, vale más que Lucía, que tiene tres?
- En el libro de la cooperativa, ¿por qué el ataque B solo se *corre* de lugar y no desaparece? ¿Qué **no** puede garantizar el libro, aunque nadie logre alterarlo?

---
#### ✍️ Tu respuesta

_Escribe aquí tus 3-5 líneas. Reemplaza este texto por tu reflexión._

---
### 🧠 Rúbrica corta

| Criterio | Puntos |
|---|---|
| Reto 1: contar por el sello y elegir cuánto esperar | 30 |
| Reto 2: importancia por rondas (PageRank) y por qué no es contar conocidos | 30 |
| Reto 3: sellos encadenados, detección del cambio y resumen de resúmenes | 25 |
| Reflexión con tus cifras | 15 |
| **Total** | **100** |

**Checklist:** ✅ ejecuté *Run All* sin errores · ✅ no quedan `...` · ✅ las tres autoverificaciones imprimen ✅ · ✅ escribí mi reflexión.

---
<div align="center">
  <p style="font-size: 0.9em; color: #78350f;">
    © 2026 <b>Universidad Santo Tomás — Seccional Tunja</b><br>
    <i>Especialización en Ciencia de Datos | Big Data (Edición Didáctica Para Dummies)</i>
  </p>
</div>